# Generate Iceberg Tables with ORC File Format on S3

This notebook creates Apache Iceberg tables that use **ORC** as the data file format
(instead of the default Parquet) and writes them directly to S3.

The output can then be loaded into Snowflake via `COPY INTO` using the companion
SQL script `02_copy_into_snowflake.sql`.

**Output S3 structure:**
```
s3://skumar-iceberg-lakehouse/iceberg/orc-demo/
├── customer_orders/
│   ├── metadata/    (Iceberg metadata JSON + Avro manifests)
│   └── data/        (*.orc files)
└── product_catalog/
    ├── metadata/
    └── data/        (*.orc files)
```

## 1. Environment Setup

In [ ]:
!pip install -q pyspark==3.5.0 findspark==2.0.1
!DEBIAN_FRONTEND=noninteractive apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-11-jdk-headless 2>&1 | tail -1

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

import findspark
findspark.init()
print(f"Spark home: {findspark.find()}")
print(f"JAVA_HOME:  {os.environ['JAVA_HOME']}")

## 2. Configuration
Update AWS credentials and S3 path below.

In [ ]:
# ── AWS CREDENTIALS ───────────────────────────────────────────────────────────
AWS_ACCESS_KEY = "<YOUR_AWS_ACCESS_KEY>"
AWS_SECRET_KEY = "<YOUR_AWS_SECRET_KEY>"
AWS_REGION     = "us-west-2"

# ── S3 OUTPUT PATH ────────────────────────────────────────────────────────────
S3_WAREHOUSE = "s3://skumar-iceberg-lakehouse/iceberg/orc-demo"

# ── ICEBERG / SPARK VERSIONS ──────────────────────────────────────────────────
ICEBERG_VERSION = "1.5.2"

print(f"S3 warehouse: {S3_WAREHOUSE}")
print(f"Region:       {AWS_REGION}")

## 3. Create Spark Session with Iceberg + Hadoop Catalog
Uses the Hadoop catalog type which writes Iceberg metadata directly to S3 (no external metastore needed).

In [ ]:
import os
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_KEY
os.environ["AWS_REGION"] = AWS_REGION

from pyspark.sql import SparkSession

ICEBERG_PACKAGES = ",".join([
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION}",
    f"software.amazon.awssdk:bundle:2.20.100",
    "org.apache.spark:spark-hadoop-cloud_2.12:3.5.0"
])

try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
      .appName("IcebergOrcGenerator")
      .master("local[*]")
      .config("spark.ui.port", "0")
      .config("spark.driver.bindAddress", "127.0.0.1")
      .config("spark.driver.host", "127.0.0.1")
      .config("spark.jars.packages", ICEBERG_PACKAGES)
      .config("spark.sql.extensions",
              "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
      .config("spark.sql.defaultCatalog", "demo_catalog")
      # Hadoop catalog writes metadata directly to S3
      .config("spark.sql.catalog.demo_catalog",
              "org.apache.iceberg.spark.SparkCatalog")
      .config("spark.sql.catalog.demo_catalog.type", "hadoop")
      .config("spark.sql.catalog.demo_catalog.warehouse", S3_WAREHOUSE)
      # S3 access
      .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY)
      .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_KEY)
      .config("spark.hadoop.fs.s3a.endpoint", f"s3.{AWS_REGION}.amazonaws.com")
      .config("spark.hadoop.fs.s3a.region", AWS_REGION)
      .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
      .config("spark.driver.extraJavaOptions", f"-Daws.region={AWS_REGION}")
      .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark session created | Version: {spark.version}")
print(f"Catalog: demo_catalog (Hadoop) -> {S3_WAREHOUSE}")

## 4. Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo_catalog.sales")
spark.sql("SHOW NAMESPACES IN demo_catalog").show(truncate=False)

## 5. Create CUSTOMER_ORDERS Table (ORC Format)
The key setting is `write.format.default` = `orc` in the table properties.

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE demo_catalog.sales.customer_orders (
        order_id     INT,
        customer_id  INT,
        product      STRING,
        amount       DECIMAL(10,2),
        order_date   DATE,
        region       STRING
    )
    USING iceberg
    TBLPROPERTIES ('write.format.default' = 'orc')
""")

spark.sql("""
    INSERT INTO demo_catalog.sales.customer_orders VALUES
      (1001, 501, 'Laptop Pro 15',       1299.99, DATE '2025-01-15', 'US-WEST'),
      (1002, 502, 'Wireless Mouse',        29.99, DATE '2025-01-16', 'US-EAST'),
      (1003, 503, 'USB-C Hub',             59.99, DATE '2025-01-17', 'US-WEST'),
      (1004, 504, '4K Monitor',           449.99, DATE '2025-02-01', 'EU-WEST'),
      (1005, 505, 'Mechanical Keyboard',   149.99, DATE '2025-02-03', 'US-WEST'),
      (1006, 506, 'Laptop Pro 15',       1299.99, DATE '2025-02-10', 'APAC'),
      (1007, 501, 'Webcam HD',             79.99, DATE '2025-03-01', 'US-EAST'),
      (1008, 507, 'Standing Desk',        599.99, DATE '2025-03-05', 'EU-WEST'),
      (1009, 508, 'Noise-Cancel Headset', 249.99, DATE '2025-03-12', 'US-WEST'),
      (1010, 509, 'Laptop Pro 15',       1299.99, DATE '2025-03-20', 'APAC')
""")

print("CUSTOMER_ORDERS created with ORC format")
spark.sql("SELECT * FROM demo_catalog.sales.customer_orders ORDER BY order_id").show(truncate=False)

## 6. Create PRODUCT_CATALOG Table (ORC Format)

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE demo_catalog.sales.product_catalog (
        product_id   INT,
        name         STRING,
        category     STRING,
        price        DECIMAL(10,2)
    )
    USING iceberg
    TBLPROPERTIES ('write.format.default' = 'orc')
""")

spark.sql("""
    INSERT INTO demo_catalog.sales.product_catalog VALUES
      (1, 'Laptop Pro 15',       'Electronics', 1299.99),
      (2, 'Wireless Mouse',      'Accessories',   29.99),
      (3, 'USB-C Hub',           'Accessories',   59.99),
      (4, '4K Monitor',          'Electronics',  449.99),
      (5, 'Mechanical Keyboard', 'Accessories',  149.99),
      (6, 'Webcam HD',           'Accessories',   79.99),
      (7, 'Standing Desk',       'Furniture',    599.99),
      (8, 'Noise-Cancel Headset','Audio',        249.99)
""")

print("PRODUCT_CATALOG created with ORC format")
spark.sql("SELECT * FROM demo_catalog.sales.product_catalog ORDER BY product_id").show(truncate=False)

## 7. Verify ORC Files on S3
Confirm that the data files are `.orc` (not `.parquet`) and Iceberg metadata exists.

In [ ]:
from pyspark.sql.functions import input_file_name

print("=== CUSTOMER_ORDERS: Data file paths ===")
(
    spark.sql("SELECT * FROM demo_catalog.sales.customer_orders")
    .select(input_file_name().alias("file_path"))
    .distinct()
    .show(truncate=False)
)

print("=== PRODUCT_CATALOG: Data file paths ===")
(
    spark.sql("SELECT * FROM demo_catalog.sales.product_catalog")
    .select(input_file_name().alias("file_path"))
    .distinct()
    .show(truncate=False)
)

## 8. Inspect Table Properties
Confirm `write.format.default = orc` is set.

In [ ]:
print("=== CUSTOMER_ORDERS properties ===")
spark.sql("SHOW TBLPROPERTIES demo_catalog.sales.customer_orders").show(truncate=False)

print("\n=== PRODUCT_CATALOG properties ===")
spark.sql("SHOW TBLPROPERTIES demo_catalog.sales.product_catalog").show(truncate=False)

## 9. List S3 Contents
Show the full directory structure written to S3.

In [ ]:
import subprocess

!pip install -q boto3
import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

bucket = "skumar-iceberg-lakehouse"
prefix = "iceberg/orc-demo/"

print(f"S3 contents under s3://{bucket}/{prefix}")
print("=" * 80)

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
for obj in response.get("Contents", []):
    size_kb = obj["Size"] / 1024
    print(f"  {obj['Key']:70s} {size_kb:>8.1f} KB")

## 10. Summary

The following Iceberg tables with **ORC data files** have been written to S3:

| Table | Rows | S3 Path |
|---|---|---|
| `customer_orders` | 10 | `s3://skumar-iceberg-lakehouse/iceberg/orc-demo/sales/customer_orders/` |
| `product_catalog` | 8 | `s3://skumar-iceberg-lakehouse/iceberg/orc-demo/sales/product_catalog/` |

**Next step**: Run `02_copy_into_snowflake.sql` in a Snowflake worksheet to load the ORC data via `COPY INTO`.

In [ ]:
spark.stop()
print("Spark session stopped. ORC generation complete.")